# 情報論A 第7回：線形分離を行う単純パーセプトロンの学習

スライドで紹介した線形分類（`sign`関数を使った単純パーセプトロンと等価）

$$
y = \mathrm{sign}(\mathbf{w}^T\mathbf{x})
$$

を、`numpy` だけで実装します。

今回の目標：

- 入力ベクトルにバイアス項 $x_0=1$ を追加する
- `sign` 関数による出力 $y \in \{-1, +1\}$ を計算する
- $y\mathbf{w}^T\mathbf{x} > 0$ なら正しく分類できている、という見方を確認する
- 誤分類したデータ点に対して、パーセプトロン学習則で重み $\mathbf{w}$ を更新する
- AND / OR / XOR ゲートを題材に、線形分離できる場合・できない場合を確認する

## 準備（変更不要）

外部データは使わないので、そのまま上から実行できます。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

# 乱数の再現性を保つ
rng = np.random.default_rng(0)


## 線形分類としての単純パーセプトロン

入力を

$$
\mathbf{x} = [1, x_1, \ldots, x_D]^T
$$

重みを

$$
\mathbf{w} = [w_0, w_1, \ldots, w_D]^T
$$

とすると、線形分類器は

$$
y = \mathrm{sign}(\mathbf{w}^T\mathbf{x})
$$

と書けます。

ここではラベルを $y \in \{-1, +1\}$ とします。`sign` 関数は次のように定義します。

$$
\mathrm{sign}(u) =
\begin{cases}
+1 & (u \geq 0) \\
-1 & (u < 0)
\end{cases}
$$

分類が正しいかどうかは、教師ラベルを $y_i$ として

$$
y_i\mathbf{w}^T\mathbf{x}_i > 0
$$

で判定できます。


In [ ]:
# ============================================================
# 基本関数
#   - add_bias: 入力データに x0 = 1 の列を追加する
#   - sign: sign関数
#   - predict: 現在の重み w でラベル y in {-1, +1} を予測する
# ============================================================

def add_bias(X_raw):
    """入力 X_raw にバイアス項 x0=1 を追加する。

    Parameters
    ----------
    X_raw : ndarray, shape (N, D)
        バイアス項を含まない入力データ。

    Returns
    -------
    X : ndarray, shape (N, D+1)
        1列目に x0=1 を追加した入力データ。
    """
    X_raw = np.asarray(X_raw, dtype=np.float64)

    # N個のデータに対して、値が1の列ベクトルを作る
    ones = np.ones((X_raw.shape[0], 1))

    # ones と X_raw を横方向に連結する
    X = np.hstack([ones, X_raw])

    return X


def sign(u):
    """sign関数。u >= 0 なら +1、それ以外なら -1 を返す。"""

    y = np.where(u >= 0, 1, -1)

    return y.astype(int)


def predict(X, w):
    """現在の重み w による予測ラベルを返す。

    Parameters
    ----------
    X : ndarray, shape (N, D+1)
        バイアス項を含む入力データ。
    w : ndarray, shape (D+1,)
        重みベクトル。

    Returns
    -------
    y : ndarray, shape (N,)
        予測ラベル。各要素は -1 または +1。
    """

    # 各データ点について u = w^T x を計算する
    u = X @ w

    # sign関数で -1/+1 に変換する
    y = sign(u)

    return y


## パーセプトロン学習則

スライドの線形分類では、以下を満たす $\mathbf{w}$ を求めることを考えました。

$$
y_i\mathbf{w}^T\mathbf{x}_i > 0
$$

つまり、

- $y_i=+1$ のデータ点では $\mathbf{w}^T\mathbf{x}_i > 0$
- $y_i=-1$ のデータ点では $\mathbf{w}^T\mathbf{x}_i < 0$

になってほしい、ということです。

誤分類している点、つまり

$$
y_i\mathbf{w}^T\mathbf{x}_i \leq 0
$$

となる点だけを使って、次のように重みを更新します。

$$
\mathbf{w}^{(new)}
\leftarrow
\mathbf{w}^{(old)} + \alpha y_i\mathbf{x}_i
$$

ここで $\alpha$ は学習率です。

直感的には、

- 本当は $+1$ なのに負側にある点では、$+\alpha\mathbf{x}_i$ を足してスコアを大きくする
- 本当は $-1$ なのに正側にある点では、$-\alpha\mathbf{x}_i$ を足してスコアを小さくする

という操作です。


In [ ]:
# ============================================================
# パーセプトロンの学習
# ============================================================

def train_perceptron(X, y_true, learning_rate=0.1, n_epochs=20, w0=None, verbose=True):
    """sign関数を使った線形分離パーセプトロンを学習する。

    Parameters
    ----------
    X : ndarray, shape (N, D+1)
        バイアス項を含む入力データ。
    y_true : ndarray, shape (N,)
        教師ラベル。各要素は -1 または +1。
    learning_rate : float
        学習率 alpha。
    n_epochs : int
        全データを何周するか。
    w0 : ndarray or None
        初期重み。Noneなら0で初期化する。
    verbose : bool
        Trueなら各epochの誤分類数を表示する。

    Returns
    -------
    w : ndarray, shape (D+1,)
        学習後の重み。
    history : dict
        各epochの誤分類数と重みを保存した辞書。
    """
    X = np.asarray(X, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=int)

    N, D_plus_1 = X.shape

    if w0 is None:
        w = np.zeros(D_plus_1, dtype=np.float64)
    else:
        w = np.asarray(w0, dtype=np.float64).copy()

    history = {
        "n_errors": [],
        "w": []
    }

    for epoch in range(n_epochs):
        n_errors = 0  # 間違えた個数カウント

        # 各学習データについて、1つずつ重みを更新する
        for xi, yi in zip(X, y_true):

            # 現在のデータ点に対するスコア u = w^T x
            u = xi @ w

            # yi * u > 0 なら正しく分類できている
            # yi * u <= 0 なら誤分類、または境界上なので更新する
            if yi * u <= 0:
                # 課題：wの更新。スライドp30あたり参照。
                w = ...

                n_errors += 1

        history["n_errors"].append(n_errors)
        history["w"].append(w.copy())

        if verbose:
            print(f"epoch {epoch+1:02d}: errors = {n_errors}, w = {w}")

        # すべて正しく分類できたら終了
        if n_errors == 0:
            break

    history["w"] = np.array(history["w"])
    history["n_errors"] = np.array(history["n_errors"])

    return w, history


## 可視化用の関数（以下変更不要）

2次元入力 $[x_1, x_2]$ に対して、学習データと決定境界を描画します。


In [ ]:
def plot_2d_data_and_boundary(X_raw, y_true, w=None, title=""):
    """2次元データとパーセプトロンの決定境界を表示する。"""

    X_raw = np.asarray(X_raw)
    y_true = np.asarray(y_true)

    plt.figure(figsize=(5, 5))

    plt.scatter(X_raw[y_true == -1, 0], X_raw[y_true == -1, 1], marker="o", label="y=-1")
    plt.scatter(X_raw[y_true == +1, 0], X_raw[y_true == +1, 1], marker="x", label="y=+1")

    if w is not None:
        # w0 + w1*x1 + w2*x2 = 0 が決定境界
        x1_min, x1_max = X_raw[:, 0].min() - 0.5, X_raw[:, 0].max() + 0.5
        x1 = np.linspace(x1_min, x1_max, 200)

        if abs(w[2]) > 1e-12:
            x2 = -(w[0] + w[1] * x1) / w[2]
            plt.plot(x1, x2, label="decision boundary")
        elif abs(w[1]) > 1e-12:
            x_const = -w[0] / w[1]
            plt.axvline(x_const, label="decision boundary")

    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.title(title)
    plt.grid(True)
    plt.axis("equal")
    plt.legend()
    plt.show()


def plot_learning_curve(history):
    """epochごとの誤分類数を表示する。"""

    plt.figure(figsize=(5, 3))
    plt.plot(np.arange(1, len(history["n_errors"]) + 1), history["n_errors"], marker="o")
    plt.xlabel("epoch")
    plt.ylabel("number of errors")
    plt.title("Learning curve")
    plt.grid(True)
    plt.show()


## 実験1：ANDゲート

ANDゲートは線形分離可能なので、単純パーセプトロンで学習できます。

ここでは、元の出力 `0` を `-1`、元の出力 `1` を `+1` として扱います。


In [ ]:
# ANDゲート
X_raw_and = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
], dtype=np.float64)

# 0 -> -1, 1 -> +1 として扱う
y_and = np.array([-1, -1, -1, +1], dtype=int)

X_and = add_bias(X_raw_and)

plot_2d_data_and_boundary(X_raw_and, y_and, title="AND gate")
print("X_and =")
print(X_and)
print("y_and =", y_and)


In [ ]:
# ANDゲートを学習する
w_and, history_and = train_perceptron(
    X_and,
    y_and,
    learning_rate=0.1,
    n_epochs=20
)

print()
print("final w:", w_and)
print("prediction:", predict(X_and, w_and))
print("ground truth:", y_and)

plot_learning_curve(history_and)
plot_2d_data_and_boundary(X_raw_and, y_and, w_and, title="AND gate: learned boundary")


## 実験2：ORゲート

ORゲートも線形分離可能です。ANDゲートと同じ学習コードで動くことを確認します。


In [ ]:
# ORゲート
X_raw_or = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
], dtype=np.float64)

# 0 -> -1, 1 -> +1 として扱う
y_or = np.array([-1, +1, +1, +1], dtype=int)

X_or = add_bias(X_raw_or)

w_or, history_or = train_perceptron(
    X_or,
    y_or,
    learning_rate=0.1,
    n_epochs=20
)

print()
print("final w:", w_or)
print("prediction:", predict(X_or, w_or))
print("ground truth:", y_or)

plot_learning_curve(history_or)
plot_2d_data_and_boundary(X_raw_or, y_or, w_or, title="OR gate: learned boundary")


## 実験3：2次元の線形分離データ

少し点数を増やして、ランダムな2次元データでも試します。


In [ ]:
# ============================================================
# ランダムな線形分離データを作る（変更不要）
# ============================================================

N = 80

# クラス -1: 左下あたり
X0 = rng.normal(loc=[-1.0, -1.0], scale=0.45, size=(N // 2, 2))

# クラス +1: 右上あたり
X1 = rng.normal(loc=[1.0, 1.0], scale=0.45, size=(N // 2, 2))

X_raw_linear = np.vstack([X0, X1])
y_linear_true = np.array([-1] * (N // 2) + [+1] * (N // 2), dtype=int)

# データの順番をシャッフル
perm = rng.permutation(N)
X_raw_linear = X_raw_linear[perm]
y_linear_true = y_linear_true[perm]

X_linear = add_bias(X_raw_linear)

plot_2d_data_and_boundary(X_raw_linear, y_linear_true, title="Linearly separable data")


In [ ]:
w_linear, history_linear = train_perceptron(
    X_linear,
    y_linear_true,
    learning_rate=0.1,
    n_epochs=30
)

y_linear_pred = predict(X_linear, w_linear)
accuracy = np.mean(y_linear_pred == y_linear_true)

print()
print("final w:", w_linear)
print("accuracy:", accuracy)

plot_learning_curve(history_linear)
plot_2d_data_and_boundary(X_raw_linear, y_linear_true, w_linear, title="Linearly separable data: learned boundary")


## 実験4：XORゲート

XORゲートは1本の直線では分離できません。  
つまり、単純パーセプトロンでは完全には学習できません。

実際に学習して、誤分類数が0にならないことを確認してみましょう。


In [ ]:
# XORゲート
X_raw_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
], dtype=np.float64)

# 0 -> -1, 1 -> +1 として扱う
y_xor = np.array([-1, +1, +1, -1], dtype=int)

X_xor = add_bias(X_raw_xor)

w_xor, history_xor = train_perceptron(
    X_xor,
    y_xor,
    learning_rate=0.1,
    n_epochs=20
)

print()
print("final w:", w_xor)
print("prediction:", predict(X_xor, w_xor))
print("ground truth:", y_xor)

plot_learning_curve(history_xor)
plot_2d_data_and_boundary(X_raw_xor, y_xor, w_xor, title="XOR gate: single perceptron")


## おさらい

1. `learning_rate` を大きくしたり小さくしたりすると、学習の様子はどう変わるか？
2. ANDゲートとORゲートで、学習後の決定境界はどう違うか？
3. XORゲートでは、なぜ誤分類数が0にならないのか？
4. バイアス項 $x_0=1$ を入れないと、どのような制限が生じるか？
5. `yi * (xi @ w) <= 0` という条件は、何を意味しているか？

## 発展課題
活性化関数にシグモイド関数、損失関数（目的関数）に二値交差エントロピーを使った実装を作ってみよう

（こうするとロジスティック回帰と呼ばれることが多い？明確な違いは曖昧な部分もある。）